# Experiment 15: Implement NoSQL Database Operations Using MongoDB in Python

**Aim:** To implement NoSQL document database operations using PyMongo / mongomock in Python on a Smart Healthcare Clinical Monitoring collection, execute CRUD operations with nested documents, build an aggregation pipeline, and contrast NoSQL with SQL schemas.

### Dataset Description
**Dataset / Document Collection:** Smart Healthcare Patient Monitoring System (`hospital_db.patients` collection, schema-flexible BSON documents featuring nested vitals objects, diagnosis arrays, and clinical attributes).
- Demonstrates flexible semi-structured JSON/BSON document paradigms.

### Theory
MongoDB is a leading document-oriented NoSQL database. Rather than storing records in rigid two-dimensional tables with fixed schemas, MongoDB organizes data into flexible BSON (Binary JSON) documents stored within collections. Key architectural characteristics include:
- Dynamic Schema: Documents in the same collection can contain different fields or varied data types.
- Embedded Documents and Arrays: Supports rich hierarchical data structures, eliminating expensive relational multi-table JOINs.
- CRUD API: Create (insert_one, insert_many), Read (find, find_one with projection and query operators like $gte, $in), Update (update_one, update_many with atomic operators like $set, $inc, $push), and Delete (delete_one, delete_many).
- Aggregation Pipeline: Multi-stage transformation framework ($match, $project, $group, $sort, $unwind) for server-side analytical aggregations.

#### 1. Establish MongoDB Connection and Initialize Collection

In [ ]:
import mongomock
# In live production: from pymongo import MongoClient; client = MongoClient('mongodb://localhost:27017/')
client = mongomock.MongoClient()
db = client['healthcare_db']
coll = db['patient_records']
coll.delete_many({}) # Reset collection
print('Connected to database healthcare_db, collection patient_records.')

#### 2. CREATE: Insert Rich Hierarchical Patient Documents

In [ ]:
patient_docs = [
    {
        'patient_id': 'P101', 'name': 'Aarav Sharma', 'age': 45, 'gender': 'Male',
        'vitals': {'bp_sys': 130, 'bp_dia': 85, 'pulse': 78},
        'diagnosis': ['Hypertension', 'Dyslipidemia'], 'department': 'Cardiology',
        'bill_amount': 14500.0, 'status': 'Admitted'
    },
    {
        'patient_id': 'P102', 'name': 'Priya Patel', 'age': 32, 'gender': 'Female',
        'vitals': {'bp_sys': 118, 'bp_dia': 76, 'pulse': 72},
        'diagnosis': ['Acute Bronchitis'], 'department': 'Pulmonology',
        'bill_amount': 8200.0, 'status': 'Discharged'
    },
    {
        'patient_id': 'P103', 'name': 'Rajesh Verma', 'age': 58, 'gender': 'Male',
        'vitals': {'bp_sys': 145, 'bp_dia': 92, 'pulse': 84},
        'diagnosis': ['Type 2 Diabetes'], 'department': 'Endocrinology',
        'bill_amount': 21000.0, 'status': 'Admitted'
    },
    {
        'patient_id': 'P104', 'name': 'Ananya Iyer', 'age': 28, 'gender': 'Female',
        'vitals': {'bp_sys': 122, 'bp_dia': 80, 'pulse': 75},
        'diagnosis': ['Sinus Arrhythmia'], 'department': 'Cardiology',
        'bill_amount': 18500.0, 'status': 'Admitted'
    },
    {
        'patient_id': 'P105', 'name': 'Vikram Malhotra', 'age': 64, 'gender': 'Male',
        'vitals': {'bp_sys': 135, 'bp_dia': 88, 'pulse': 80},
        'diagnosis': ['Chronic Asthma'], 'department': 'Pulmonology',
        'bill_amount': 9500.0, 'status': 'Discharged'
    }
]
res = coll.insert_many(patient_docs)
print(f'Successfully inserted {len(res.inserted_ids)} patient documents.')

#### 3. READ, UPDATE, and DELETE Operations

In [ ]:
# READ with filter and projection
print('Cardiology Patients:')
for doc in coll.find({'department': 'Cardiology'}, {'_id': 0, 'name': 1, 'vitals': 1}):
    print(doc)

# UPDATE with atomic $set and $inc
upd = coll.update_one({'patient_id': 'P101'}, {'$set': {'status': 'Discharged'}, '$inc': {'bill_amount': 1200.0}})
print(f'Updated matched: {upd.matched_count}, modified: {upd.modified_count}')

# DELETE one record
del_res = coll.delete_one({'patient_id': 'P105'})
print(f'Deleted documents count: {del_res.deleted_count}')
print(f'Remaining documents: {coll.count_documents({})}')

#### 4. Multi-Stage Aggregation Pipeline and SQL Contrast

In [ ]:
pipeline = [
    {'$match': {'bill_amount': {'$gt': 5000}}},
    {'$group': {
        '_id': '$department',
        'total_revenue': {'$sum': '$bill_amount'},
        'avg_revenue': {'$avg': '$bill_amount'},
        'patient_count': {'$sum': 1}
    }},
    {'$sort': {'total_revenue': -1}}
]
agg_results = list(coll.aggregate(pipeline))
print('\nAggregation Pipeline Results by Department:')
for r in agg_results:
    print(f"Dept: {r['_id']:14s} | Patients: {r['patient_count']} | Total: ${r['total_revenue']:.2f} | Avg: ${r['avg_revenue']:.2f}")

### Observations
1. 5 multi-field patient documents containing embedded vital objects and diagnosis arrays were successfully ingested.
2. Querying with field projections and atomic updates ($set and $inc) modified patient status and bill amounts without schema validation errors.
3. The multi-stage aggregation pipeline ($match -> $group -> $sort) aggregated clinical billings by hospital department:
   - Cardiology: 2 patients, Total Revenue = $34,200.00, Avg = $17,100.00.
   - Endocrinology: 1 patient, Total Revenue = $21,000.00, Avg = $21,000.00.
   - Pulmonology: 1 patient, Total Revenue = $8,200.00, Avg = $8,200.00.
4. MongoDB documents allow denormalized clinical storage without requiring multi-table relational foreign-key joins.

### Result
MongoDB NoSQL database operations were successfully demonstrated from Python. Dynamic schema adaptability and native JSON/BSON structures support agile analytics on semi-structured clinical data, while aggregation pipelines provide SQL GROUP BY equivalents.